In [ ]:
# importing basic libraries
import numpy as np
import pandas as pd
import xarray as xr
import glob

# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# machine learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [ ]:
ssta_files = glob.glob("sample_data/ct5km_ssta_v3.1_*.nc")
dhw_files = glob.glob("sample_data/ct5km_dhw_v3.1_*.nc")
sst_files = glob.glob("sample_data/coraltemp_v3.1_*.nc")

print(len(ssta_files), len(dhw_files), len(sst_files))

In [ ]:
# SSTA
ssta = xr.concat([xr.open_dataset(f) for f in ssta_files], dim="time")

# DHW
dhw = xr.concat([xr.open_dataset(f) for f in dhw_files], dim="time")

# SST
sst = xr.concat([xr.open_dataset(f) for f in sst_files], dim="time")

In [ ]:
for df_temp in [ssta_df, dhw_df, sst_df]:
    for col in df_temp.select_dtypes(include=['float64']).columns:
        df_temp[col] = df_temp[col].astype('float32')

In [ ]:
df1 = pd.merge(ssta_df, dhw_df, on=['time','lat','lon'], how="inner")

del ssta_df
del dhw_df
import gc
gc.collect()

df = pd.merge(df1, sst_df, on=['time','lat','lon'], how="inner")

del df1
del sst_df
gc.collect()

In [ ]:
ssta_df = ssta['ssta'].to_dataframe().reset_index()
dhw_df = dhw['dhw'].to_dataframe().reset_index()
sst_df = sst['sst'].to_dataframe().reset_index()

ssta_df.rename(columns={'ssta': 'ssta'}, inplace=True)
sst_df.rename(columns={'sst': 'sst'}, inplace=True)
dhw_df.rename(columns={'dhw': 'dhw'}, inplace=True)

In [ ]:
# merging datasets
df = pd.merge(ssta_df, dhw_df, on=['time','lat','lon'])

print("Shape:", df.shape)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()
print("After removing nulls:", df.shape)

In [ ]:
df = df.drop_duplicates()

In [ ]:
Q1 = df['ssta'].quantile(0.25)
Q3 = df['ssta'].quantile(0.75)

IQR = Q3 - Q1

low = Q1 - 1.5 * IQR
high = Q3 + 1.5 * IQR

df = df[(df['ssta'] >= low) & (df['ssta'] <= high)]

In [ ]:
print(df.describe())

In [ ]:
sns.histplot(df['ssta'], bins=40)
plt.title("SSTA Distribution")
plt.show()

In [ ]:
sns.boxplot(x=df['ssta'])
plt.title("Outliers Check")
plt.show()

In [ ]:
sample = df.sample(20000)

sns.scatterplot(x='lon', y='lat', hue='ssta',
                data=sample, palette='coolwarm', s=5)

plt.title("Ocean Temperature Map")
plt.show()

In [ ]:
time_data = df.groupby('time')['ssta'].mean()

time_data.plot()
plt.title("Temperature Over Time")
plt.xlabel("Time")
plt.ylabel("SSTA")
plt.show()

In [ ]:
sns.heatmap(df[['ssta','dhw']].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
X = df[['lat','lon','dhw']]
y = df['ssta']

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, pred)
print("Mean Squared Error:", mse)

In [ ]:
plt.scatter(y_test[:1000], pred[:1000])
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Prediction vs Actual")
plt.show()